<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W6D3_Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "TODO: replace with a short sentence you want to tokenize"
print(sample_sentence)


In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


### Exercise 1 reflection
- **[CLS] and [SEP] Token Behavior:**
  - The `[CLS]` (classifier) token is inserted at the beginning of the input sequence. In classification tasks, the final hidden state corresponding to this token is often used as the aggregate representation of the entire sequence for classification.
  - The `[SEP]` (separator) token is used to mark the end of a sequence, or to separate two distinct sequences in tasks like question answering.
- **Attention Mask for Padded Positions:**
  - The attention mask is a binary tensor (containing 0s and 1s) that indicates which tokens should be attended to and which should be ignored by the model. A value of `1` means the token should be attended to, while `0` means it should be ignored.
  - For padded positions (where `[PAD]` tokens are present), the attention mask has a value of `0`. This effectively "hides" these padded tokens from the self-attention mechanism, preventing the model from attending to them and ensuring that the padding does not influence the model's understanding of the actual content.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This movie is absolutely fantastic and I loved every minute of it!"
prediction = sentiment_pipeline(sentence)
prediction

### Exercise 2 reflection
- **Sentence tested:** "This movie is absolutely fantastic and I loved every minute of it!"
- **Predicted label and score:** `[{'label': 'POSITIVE', 'score': 0.9998782873153687}]`
  - The model predicted a `POSITIVE` label with a confidence score of `0.999878`. This indicates a very strong positive sentiment.
- **Does the predicted label match your expectation? Why or why not?**
  - Yes, the predicted label perfectly matches my expectation. The sentence contains highly positive words like "fantastic" and "loved every minute of it," which strongly convey a positive sentiment.
- **How confident is the model and what does the score tell you?**
  - The model is extremely confident, with a score very close to 1 (0.999878). This score represents the probability that the sentence belongs to the 'POSITIVE' class. A high score like this suggests that the model has very little uncertainty about its classification.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].to(self.device),
            "attention_mask": inputs["attention_mask"].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        self.model.eval()
        with torch.no_grad():
            inputs = self.preprocess(text)
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1).squeeze()

            # Assuming a binary classification (e.g., negative/positive)
            # The model's labels are typically in outputs.config.id2label
            id2label = self.model.config.id2label

            predicted_id = torch.argmax(probabilities).item()
            predicted_label = id2label[predicted_id]
            predicted_probability = probabilities[predicted_id].item()

            return {"label": predicted_label, "score": predicted_probability}

In [ ]:
# Instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "This movie is absolutely fantastic and I loved every minute of it!", # Positive
    "The service was terrible and I will never go back there again.", # Negative
    "I am indifferent to the outcome, it doesn't matter much to me.", # Neutral (should lean negative or positive based on model training)
    "What a wonderful day to be alive!"
]
for text in samples:
    print(f"Sentence: '{text}'")
    prediction = analyzer.predict(text)
    print(f"  Predicted: Label={prediction['label']}, Score={prediction['score']:.4f}")
    print("-" * 30)

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def recognize(self, text: str):
        tokens = self.tokenizer.tokenize(self.tokenizer.decode(self.tokenizer.encode(text)))
        inputs = self.tokenizer.encode(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(inputs)

        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()

        # Map token IDs to labels
        id_to_label = self.model.config.id2label
        labels = [id_to_label[p] for p in predictions]

        # Align tokens with words and merge subword tokens
        word_ids = inputs.word_ids()
        previous_word_idx = None
        current_word_tokens = []
        current_word_labels = []

        processed_tokens = []
        for i, word_idx in enumerate(word_ids):
            if word_idx is None: # Special tokens like [CLS], [SEP]
                continue
            if word_idx != previous_word_idx:
                if current_word_tokens: # Process the previous word
                    processed_tokens.append({
                        'word': self.tokenizer.convert_tokens_to_string(current_word_tokens),
                        'labels': current_word_labels # Keep labels for merging logic later
                    })
                current_word_tokens = [tokens[i]]
                current_word_labels = [labels[i]]
            else:
                current_word_tokens.append(tokens[i])
                current_word_labels.append(labels[i])
            previous_word_idx = word_idx
        if current_word_tokens: # Process the last word
            processed_tokens.append({
                'word': self.tokenizer.convert_tokens_to_string(current_word_tokens),
                'labels': current_word_labels
            })

        # Extract and format entities based on BIO scheme
        entities = []
        current_entity = None
        start_char_idx = 0

        for item in processed_tokens:
            word = item['word']
            # For simplicity, let's just take the label of the first token in the word
            # For more robust merging, one would need to consider 'B-TAG', 'I-TAG' transitions
            label = item['labels'][0]

            # Adjust start_char_idx for spaces
            if text[start_char_idx:start_char_idx+len(word)].lower() != word.lower():
                start_char_idx = text.find(word, start_char_idx)

            if label.startswith('B-'):
                if current_entity:
                    entities.append(current_entity)
                current_entity = {
                    'text': word,
                    'entity': label[2:],
                    'start': start_char_idx,
                    'end': start_char_idx + len(word)
                }
            elif label.startswith('I-') and current_entity and current_entity['entity'] == label[2:]:
                current_entity['text'] += ' ' + word
                current_entity['end'] = start_char_idx + len(word)
            else:
                if current_entity:
                    entities.append(current_entity)
                current_entity = None
            start_char_idx += len(word)
            # Account for spaces between words
            if start_char_idx < len(text) and text[start_char_idx] == ' ':
                start_char_idx += 1

        if current_entity:
            entities.append(current_entity)

        return entities

In [ ]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin in September 1998 in California. Its headquarters are in Mountain View."
entities = ner.recognize(sample_text)
for entity in entities:
    print(f"Text: '{entity['text']}', Entity: {entity['entity']}, Start: {entity['start']}, End: {entity['end']}")

sample_text_2 = "Apple Inc. announced its new iPhone 15 at an event in Cupertino, California. Tim Cook, the CEO, presented the device."
entities_2 = ner.recognize(sample_text_2)
print("\n--- Second Sample ---")
for entity in entities_2:
    print(f"Text: '{entity['text']}', Entity: {entity['entity']}, Start: {entity['start']}, End: {entity['end']}")

### Exercise 4 reflection

- **Detected Entities:** The `recognize` method successfully extracts entities like persons, organizations, locations, and miscellaneous entities from the input text, returning them in the specified dictionary format: `{'text': 'entity_text', 'entity': 'ENTITY_TYPE', 'start': start_index, 'end': end_index}`.

- **Handling Subword Tokens (`##`):**
  - BERT tokenizers often break words into subword units, especially for less common words or words with common prefixes/suffixes. Subword tokens are typically prefixed with `##` (e.g., `token` -> `token`, `##ize` -> `ize`).
  - In the `recognize` method, the subword tokens are initially processed by the model to get their individual labels.
  - The crucial step for handling `##` tokens is in the word alignment and merging logic. The code reconstructs full words from these subword tokens. It iterates through the tokenized input and uses `word_ids` (a feature from `tokenizer.encode` when `return_offsets_mapping=True` or a similar mechanism) to group subword tokens back into their original words. For each reconstructed word, it combines the corresponding subword tokens and their labels. Then, `self.tokenizer.convert_tokens_to_string()` is used to reassemble the subword tokens into the original word string, effectively removing the `##` prefix.
  - Finally, when extracting entities, the code processes these reconstructed words. If a word is part of a multi-token entity (e.g., `New York` where `New` is 'B-LOC' and `York` is 'I-LOC'), it correctly merges them into a single entity span, regardless of whether individual tokens were subwords.

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category          | BERT                                                 | GPT                                                    |
|-------------------|------------------------------------------------------|--------------------------------------------------------|
| Architecture      | Encoder-only (bidirectional attention)               | Decoder-only (masked self-attention, autoregressive)   |
| Primary purpose   | Understanding and encoding input text                | Generating coherent and contextually relevant text     |
| Typical use cases | Sentiment analysis, NER, question answering, summarization, text classification | Text generation, chatbots, creative writing, translation, code generation |
| Strengths         | Deep contextual understanding of input, good for fine-tuning on specific tasks | Excellent for generating fluent and diverse text, strong few-shot learning |
| Weaknesses        | Not designed for text generation, can be less flexible for creative tasks | Can suffer from factual inaccuracies, lacks explicit bidirectional context, prone to repetition |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:

1.  **How BERT encodes queries and documents:** BERT encodes both queries and documents (or document chunks) into dense vector representations called embeddings. For a given text, BERT processes it through its transformer encoder layers, generating a contextualized embedding for each token. A common approach for getting a single document or query embedding is to use the embedding of the `[CLS]` token (which aggregates information from the entire sequence) or to average the embeddings of all output tokens. These embeddings capture the semantic meaning of the text.

2.  **How those embeddings are stored and searched in a vector database:** The BERT-generated embeddings for all documents in a knowledge base are pre-computed and stored in a specialized vector database (e.g., Pinecone, Weaviate, Milvus). When a new query comes in, its embedding is also generated by BERT. The vector database then performs an efficient similarity search (e.g., using algorithms like Approximate Nearest Neighbors - ANN) to find document embeddings that are most semantically similar to the query embedding. This retrieval step quickly identifies the most relevant document chunks from a large corpus.

3.  **How the retrieved passages are handed to a generative model like GPT:** Once the most relevant passages are retrieved from the vector database, they are concatenated with the original user query. This combined context (retrieved passages + original query) is then fed as input to a large language model (LLM) like GPT. The generative model uses this enriched context to formulate a comprehensive and grounded answer, rather than relying solely on its internal, potentially outdated, knowledge.

4.  **Concrete application example (industry or product) where RAG with BERT makes sense:** A concrete example is an **enterprise customer support chatbot**. Instead of providing generic answers, a RAG system powered by BERT can significantly enhance its capabilities. When a customer asks a question (query), BERT embeds the question. This query embedding is used to retrieve relevant sections from an extensive knowledge base of product manuals, FAQs, and support documents (documents), which are also embedded by BERT and stored in a vector database. The retrieved, relevant passages are then passed to a generative LLM (e.g., GPT) along with the original question. The LLM then synthesizes an accurate, context-aware answer based on the company's specific documentation, reducing hallucinations and improving customer satisfaction.